In [1]:
from ultralytics import YOLO
import torch
import os


In [3]:
from ultralytics import YOLO
import torch

# إعداد GPU
device = 0 if torch.cuda.is_available() else 'cpu'

# 1. تغيير الموديل لـ Small (أخف وأسرع)
model = YOLO('yolov8s.pt') 

print("🚀 Starting Optimized Training...")

results = model.train(
    data='formal_data_absolute.yaml',
    name='formal_wear_small_fast',
    
    # --- تعديلات السرعة ---
    epochs=50,             # 50 دورة كافية جداً للموديل الـ Small
    patience=10,           # لو مفيش تحسن في 10 دورات اقفل
    batch=16,              # الـ Small يسمح بباتش أكبر (أسرع في التدريب)
    imgsz=512,             # تصغير الصورة قليلاً لتسريع الكارت (بدل 640)
    cache=True,            # تحميل الصور في الرامات للسرعة القصوى
    workers=4,
    device=device,
    
    # --- الاحتفاظ بالتحسينات ---
    augment=True,
    hsv_v=0.4,             # تغيير الإضاءة مهم
    degrees=10.0,
    fliplr=0.5
)

print("✅ Done! Saved in runs/detect/formal_wear_small_fast/weights/best.pt")

🚀 Starting Optimized Training...
Ultralytics 8.4.6  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=formal_data_absolute.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=formal_wear_small_fast, nbs=64, nms=False, opset=None, optimize=False, optimizer=a

In [1]:
from ultralytics import YOLO

# 1. تحميل أفضل نسخة وصل لها الموديل
# تأكدي أن المسار صحيح كما فعلنا سابقاً
model = YOLO('C:/runs/detect/formal_wear_small_fast/weights/best.pt')

# 2. تشغيل اختبار الدقة على بيانات الاختبار (Validation Set)
metrics = model.val()

# 3. طباعة النتائج بشكل مقروء
print(f"Mean Average Precision (mAP50): {metrics.box.map50:.3f}")
print(f"Precision: {metrics.box.mp:.3f}")
print(f"Recall: {metrics.box.mr:.3f}")

Ultralytics 8.4.6  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 6144MiB)
Model summary (fused): 73 layers, 11,127,132 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access  (ping: 0.90.2 ms, read: 199.049.4 MB/s, size: 59.8 KB)
val: Scanning C:\Users\anesr\OneDrive\Documents\GP Project models\datasets\formal_wear_filtered\labels\val.cache... 711 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 711/711  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 45/45 7.4it/s 6.1s0.1s
                   all        711       1204      0.854      0.848      0.923       0.78
                 Shirt        547        579      0.891      0.706      0.863      0.697
                Jacket        298        308      0.861      0.807      0.886      0.787
                   Tie          3          3      0.762          1      0.995       0.78
                 Pants        313        314      0.902      0

In [15]:
import cv2
from ultralytics import YOLO
import numpy as np
import os

model = YOLO('C:/runs/detect/formal_wear_small_fast/weights/best.pt')
video_path = 'C:/Users/anesr/Downloads/Video Project 7.mp4' # مسار الفيديو
cap = cv2.VideoCapture(video_path)

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
num_samples = 15 
indices = np.linspace(0, total_frames-1, num_samples, dtype=int)

formal_points = 0    # نقاط الفورمال
frames_analyzed = 0  

print(f"--- Starting STRICT Analysis ---")

for i in indices:
    cap.set(cv2.CAP_PROP_POS_FRAMES, i)
    ret, frame = cap.read()
    if not ret: continue

    # نثق فقط في النتائج القوية
    results = model(frame, verbose=False, conf=0.5)
    
    frame_points = 0
    detected_items = []

    for r in results:
        for box in r.boxes:
            cls_id = int(box.cls[0])
            label = model.names[cls_id].lower()
            conf = float(box.conf[0])
            
            # --- 🔥 التعديل الجوهري في اللوجيك 🔥 ---
            
            if label in ['jacket', 'suit', 'tie']:
                frame_points += 5        # الجاكيت يعطي نقاط عالية جداً (فورمال مؤكد)
                detected_items.append(label)
                
            elif label == 'shirt':
                frame_points += 1        # القميص يعطي نقطة واحدة فقط (قد يكون تي شيرت)
                detected_items.append(label)
            
            # البنطلون لا يؤثر في الحكم لأنه لا يظهر غالباً في المقابلات
            # elif label == 'pants': 
            #     pass 

    # اللوجيك الجديد:
    # لكي يعتبر الفريم "فورمال"، يجب أن يجمع 3 نقاط على الأقل
    # يعني: جاكيت واحد (5 نقاط) = فورمال
    # أو: 3 قمصان (3 نقاط) = فورمال (احتمال ضعيف)
    # القميص لوحده (1 نقطة) = ليس فورمال بما فيه الكفاية
    
    if frame_points >= 3:
        formal_points += 1
        print(f"Frame {i}: ✅ Formal (Score: {frame_points}) - Found: {detected_items}")
    else:
        # هنا القميص لوحده سيعتبر Casual
        print(f"Frame {i}: ⚠️ Casual/Low Confidence (Score: {frame_points}) - Found: {detected_items}")
        
    frames_analyzed += 1

cap.release()

# حساب النتيجة النهائية
if frames_analyzed > 0:
    final_ratio = formal_points / frames_analyzed
else:
    final_ratio = 0

status = "Formal" if final_ratio >= 0.5 else "Casual"

print(f"\n--- Final Verdict ---")
print(f"Formal Ratio: {final_ratio:.2f}")
print(f"Final Assessment: {status}")

if status == "Formal":
    print("✅ Great! You are wearing professional attire (Jacket/Tie detected).")
else:
    print("❌ Casual Attire Detected. (Wearing just a shirt might not be enough, wear a Jacket for better results).")

--- Starting STRICT Analysis ---
Frame 0: ⚠️ Casual/Low Confidence (Score: 1) - Found: ['shirt']
Frame 193: ⚠️ Casual/Low Confidence (Score: 1) - Found: ['shirt']
Frame 387: ⚠️ Casual/Low Confidence (Score: 1) - Found: ['shirt']
Frame 581: ⚠️ Casual/Low Confidence (Score: 1) - Found: ['shirt']
Frame 775: ⚠️ Casual/Low Confidence (Score: 1) - Found: ['shirt']
Frame 968: ⚠️ Casual/Low Confidence (Score: 1) - Found: ['shirt']
Frame 1162: ⚠️ Casual/Low Confidence (Score: 1) - Found: ['shirt']
Frame 1356: ⚠️ Casual/Low Confidence (Score: 1) - Found: ['shirt']
Frame 1550: ⚠️ Casual/Low Confidence (Score: 1) - Found: ['shirt']
Frame 1744: ⚠️ Casual/Low Confidence (Score: 1) - Found: ['shirt']
Frame 1937: ⚠️ Casual/Low Confidence (Score: 1) - Found: ['shirt']
Frame 2131: ⚠️ Casual/Low Confidence (Score: 1) - Found: ['shirt']
Frame 2325: ⚠️ Casual/Low Confidence (Score: 1) - Found: ['shirt']
Frame 2519: ⚠️ Casual/Low Confidence (Score: 1) - Found: ['shirt']
Frame 2713: ⚠️ Casual/Low Confidence 